In [1]:
import pandas as pd
import numpy as np

In [2]:
renta_raw = pd.read_csv("../data_raw/renta_media.csv", sep=";", low_memory=False)

In [3]:
renta_raw.head()

,Municipios,Distritos,Secciones,Indicadores de renta media,Periodo,Total
0,01001 Alegría-Dulantzi,NaN,NaN,Renta neta media por persona,2023,16.429
1,01001 Alegría-Dulantzi,NaN,NaN,Renta neta media por persona,2022,15.116
2,01001 Alegría-Dulantzi,NaN,NaN,Renta neta media por persona,2021,14.647
3,01001 Alegría-Dulantzi,NaN,NaN,Renta neta media por persona,2020,13.969
4,01001 Alegría-Dulantzi,NaN,NaN,Renta neta media por persona,2019,14.299


In [4]:
renta_raw.shape

(3009312, 6)

In [5]:
renta_raw.dtypes

Municipios                    object
Distritos                     object
Secciones                     object
Indicadores de renta media    object
Periodo                        int64
Total                         object
dtype: object

## Limpieza de estructura del dataset

In [6]:
# Normalizar nombres de columnas
renta_raw.columns = (
    renta_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [7]:
# Revisión de valores nulos
renta_raw.isna().sum()

municipios                          0
distritos                      439506
secciones                     1007424
indicadores_de_renta_media          0
periodo                             0
total                           85270
dtype: int64

In [8]:
# Nos quedamos solo con las filas a nivel municicpio
# En el nivel municipio, distritos y secciones están vacios
renta = renta_raw[
    (renta_raw["distritos"].isna()) &
    (renta_raw["secciones"].isna())
].copy()

In [9]:
renta.shape

(439506, 6)

In [10]:
# Eliminamos columnas que no sirven
renta = renta.drop(columns=['distritos', 'secciones'])

In [11]:
# Comprobamos que indicadores nos da la tabla
renta['indicadores_de_renta_media'].unique()

array(['Renta neta media por persona', 'Renta neta media por hogar',
       'Media de la renta por unidad de consumo',
       'Mediana de la renta por unidad de consumo',
       'Renta bruta media por persona', 'Renta bruta media por hogar'],
      dtype=object)

In [12]:
# Solo necesitamos "Renta neta media por persona"
renta = renta[renta['indicadores_de_renta_media'] == "Renta neta media por persona"]

In [13]:
renta['indicadores_de_renta_media'].unique()

array(['Renta neta media por persona'], dtype=object)

In [14]:
# Ya hemos filtrado por el indicador necesario, por lo que podemos eliminar la columna del indicador ya que no aporta información
renta = renta.drop(columns=['indicadores_de_renta_media'])

In [15]:
renta.head()

,municipios,periodo,total
0,01001 Alegría-Dulantzi,2023,16.429
1,01001 Alegría-Dulantzi,2022,15.116
2,01001 Alegría-Dulantzi,2021,14.647
3,01001 Alegría-Dulantzi,2020,13.969
4,01001 Alegría-Dulantzi,2019,14.299


In [16]:
# Separamos código y nombre del municipio
renta[['codigo_municipio', 'municipio']] = renta['municipios'].str.split(" ", n=1, expand=True)

In [17]:
# Reordenamos columnas
cols = ["codigo_municipio", "municipio"] + [col for col in renta.columns if col not in ["codigo_municipio", "municipio"]]
renta = renta[cols]

In [18]:
renta.head()

,codigo_municipio,municipio,municipios,periodo,total
0,01001,Alegría-Dulantzi,01001 Alegría-Dulantzi,2023,16.429
1,01001,Alegría-Dulantzi,01001 Alegría-Dulantzi,2022,15.116
2,01001,Alegría-Dulantzi,01001 Alegría-Dulantzi,2021,14.647
3,01001,Alegría-Dulantzi,01001 Alegría-Dulantzi,2020,13.969
4,01001,Alegría-Dulantzi,01001 Alegría-Dulantzi,2019,14.299


In [19]:
# Eliminamos la columna original "municipios"
renta = renta.drop(columns=['municipios'])

In [20]:
renta.head()

,codigo_municipio,municipio,periodo,total
0,01001,Alegría-Dulantzi,2023,16.429
1,01001,Alegría-Dulantzi,2022,15.116
2,01001,Alegría-Dulantzi,2021,14.647
3,01001,Alegría-Dulantzi,2020,13.969
4,01001,Alegría-Dulantzi,2019,14.299


In [21]:
# Arrelgamos el código de municipio (5 dígitos) para asegurar consistencia
renta['codigo_municipio'] = renta['codigo_municipio'].astype(str).str.zfill(5)

In [22]:
# Revisamos nulos
renta.isna().sum()

codigo_municipio      0
municipio             0
periodo               0
total               180
dtype: int64

In [23]:
# Inspeccionamos los nulos
renta[renta['total'].isna()].head(20)

,codigo_municipio,municipio,periodo,total
152442,04039,Darrical,2023,NaN
152443,04039,Darrical,2022,NaN
152444,04039,Darrical,2021,NaN
152445,04039,Darrical,2020,NaN
152446,04039,Darrical,2019,NaN
152447,04039,Darrical,2018,NaN
152448,04039,Darrical,2017,NaN
152449,04039,Darrical,2016,NaN
152450,04039,Darrical,2015,NaN
604589,09243,Padilla de Arriba,2018,NaN


In [24]:
renta[renta["total"].isna()]["periodo"].value_counts().sort_index()

periodo
2015    102
2016     15
2017     14
2018     10
2019      8
2020      8
2021      8
2022      8
2023      7
Name: count, dtype: int64

In [25]:
# Comprobamos tipo antes de convertir
renta['total'].dtype

dtype('O')

In [26]:
# Limpiamos el separador de miles
renta["total"] = (
    renta["total"]
    .astype(str)
    .str.replace(".", "", regex=False)
)

In [27]:
# Convertimos a númerico
renta['total'] = pd.to_numeric(renta['total'], errors="coerce")

In [29]:
# Comprobamos nulos despues de convertir
renta['total'].isna().sum()

np.int64(7522)

In [30]:
# Eliminamos filas sin dato de renta
renta = renta.dropna(subset=['total'])

In [31]:
# Renombramos columnas para mantener consistencia con el resto de datasets
renta = renta.rename(columns={
    'total': 'renta_media'
})

In [32]:
# Reordenamos columnas finales
renta = renta[['codigo_municipio', 'municipio', 'periodo', 'renta_media']]

In [33]:
# Ordenamos 
renta = renta.sort_values(['codigo_municipio', 'periodo'])

In [34]:
# Comprobaciones finales
renta.head()

,codigo_municipio,municipio,periodo,renta_media
8,01001,Alegría-Dulantzi,2015,12936.0
7,01001,Alegría-Dulantzi,2016,13086.0
6,01001,Alegría-Dulantzi,2017,13281.0
5,01001,Alegría-Dulantzi,2018,13361.0
4,01001,Alegría-Dulantzi,2019,14299.0


In [35]:
renta.shape, renta.dtypes, renta.isna().sum()

((65729, 4),
 codigo_municipio     object
 municipio            object
 periodo               int64
 renta_media         float64
 dtype: object,
 codigo_municipio    0
 municipio           0
 periodo             0
 renta_media         0
 dtype: int64)

## Exportamos dataset limpio

In [37]:
renta.to_csv("../data_processed/renta_limpio.csv", index=False)